# 05_sensor_gold

**Purpose:** Build the sensor Gold tables and reconcile them with Silver, Bronze, the simulation and Petrinex (DEC-034).
All readings are simulated; the reported volumes are real Petrinex values from `gold_fact_production`. The wells that
are flagged were set to differ on purpose in `config/sensor.json`: this tests the detection, it is not a finding about
the real data.

**Input tables:** `silver_sensor_readings`, `sensor_silver_rejects`, the two Bronze sensor tables, `pipeline_run_log`,
`gold_fact_production`; `Files/config/sensor.json`; the backfill truth file.

**Output tables:**
- `gold_sensor_hourly` (overwrite): one row per tag and hour, with values from Good readings only and counts by quality
- `gold_downtime_events` (overwrite): PI event frame style, one row per shut-in of the 2025-11 history, with its hours
  and estimated lost volume
- `gold_sensor_reconciliation` (overwrite): per well for 2025-11, sensor volume against Petrinex, with the
  `needs_investigation` flag
- `sensor_reconciliation_results` (append): the 4 tests below; the notebook stops after writing them if any test fails. Not `reconciliation_results`, which holds only the Petrinex pipeline's tests (DEC-034)

**Reconciliation tests:**
1. `bronze_equals_silver_received_plus_rejects`: rows in both Bronze tables = received rows in Silver + rejects of this run
2. `gold_hourly_good_count_and_sum_match_silver`: tags whose Good count or sum of Good values differs from Silver (0)
3. `downtime_events_match_simulation_truth`: events differing from the truth file, in either direction (0)
4. `flagged_wells_match_configured_anomalies`: wells flagged or configured, but not both (0)

Run with no stream sending, because test 1 counts Bronze as it is now (DEC-033: the Eventstream is deactivated between
runs).

**Run order:** step 2 of 2 in `pl_sensor_hourly`: 04_sensor_silver -> **05_sensor_gold**.


In [ ]:
import json
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")
with open("/lakehouse/default/Files/config/sensor.json") as fh:
    CFG = json.load(fh)
L = CFG["layers"]
DT = CFG["downtime"]
TZ = CFG["month"]["timezone"]

silver = spark.table(L["silver_table"]).cache()

# Reuse the run ID of the 04_sensor_silver run, and stop if that run did not finish
run_ids = [r[0] for r in silver.select("_run_id").distinct().collect()]
if len(run_ids) != 1:
    raise ValueError(f"{L['silver_table']} must hold one _run_id, found {run_ids}. Run 04_sensor_silver first.")
RUN_ID = run_ids[0]
finished = (spark.table("pipeline_run_log")
    .filter((F.col("run_id") == RUN_ID) & (F.col("source") == "sensor") & (F.col("status") == "SUCCESS")).count())
if not finished:
    raise ValueError(f"Run {RUN_ID} of 04_sensor_silver did not finish (no SUCCESS in pipeline_run_log). Rerun it.")
print("RUN_ID:", RUN_ID, "| Silver rows:", silver.count())


In [ ]:
# Hourly values from Good readings only; the counts show how much of each hour they cover
good = F.col("quality") == "Good"
hourly = (silver.withColumn("hour_utc", F.date_trunc("hour", "ts_utc"))
    .groupBy("tag_name", "hour_utc", "source", "well_key", "facility_id", "product", "tag_code", "unit")
    .agg(F.avg(F.when(good, F.col("value"))).alias("avg_value"),
         F.min(F.when(good, F.col("value"))).alias("min_value"),
         F.max(F.when(good, F.col("value"))).alias("max_value"),
         F.sum(F.when(good, F.col("value"))).alias("sum_good_value"),
         F.sum(F.when(good, 1).otherwise(0)).alias("good_count"),
         F.sum(F.when(F.col("quality") == "Questionable", 1).otherwise(0)).alias("questionable_count"),
         F.sum(F.when(F.col("quality") == "Bad", 1).otherwise(0)).alias("bad_count"),
         F.count(F.lit(1)).alias("reading_count"))
    .withColumn("local_day", F.to_date(F.from_utc_timestamp("hour_utc", TZ)))
    .withColumn("_run_id", F.lit(RUN_ID)))
hourly.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(L["hourly_table"])


In [ ]:
# One row per well and hour of the history: the flow reading beside the run status (sensor/reconcile.py and
# sensor/downtime.py hold the same rules)
history = silver.filter(F.col("stream_run_id").isNull())
flow_reading = history.filter(F.col("tag_code").startswith("FLOW_")).select(
    "well_key", "facility_id", "product", "ts_utc", "local_day", "value", "quality")
run_reading = history.filter(F.col("tag_code") == "RUN").select(
    "well_key", "ts_utc", F.col("value").alias("run_value"), F.col("quality").alias("run_quality"))
flow_off = F.coalesce(*[F.when(F.col("product") == p, F.lit(float(v))) for p, v in DT["flow_off_below"].items()])
well_hours = (flow_reading.join(run_reading, ["well_key", "ts_utc"], "left")
    .withColumn("good_flow", F.col("quality") == "Good")
    .withColumn("shut_in", F.coalesce((F.col("run_quality") == "Good") & (F.col("run_value") == 0), F.lit(False)))
    .withColumn("running_good", F.col("good_flow") & ~F.col("shut_in") & (F.col("value") > flow_off))
    .withColumn("running_rate", F.when(F.col("running_good"), F.col("value")))
    .cache())
print("well hours:", well_hours.count())


In [ ]:
# Downtime events: runs of at least detect_min_hours consecutive shut-in hours
by_well = Window.partitionBy("well_key").orderBy("ts_utc")
islands = (well_hours
    .withColumn("_all", F.row_number().over(by_well))
    .withColumn("_same", F.row_number().over(Window.partitionBy("well_key", "shut_in").orderBy("ts_utc")))
    .filter("shut_in")
    .groupBy("well_key", "facility_id", "product", F.col("_all") - F.col("_same"))
    .agg(F.min("ts_utc").alias("start_utc"), F.count(F.lit(1)).alias("duration_hours"))
    .filter(F.col("duration_hours") >= DT["detect_min_hours"])
    .withColumn("end_utc", F.timestamp_seconds(F.unix_timestamp("start_utc") + F.col("duration_hours") * 3600))
    .select("well_key", "facility_id", "product", "start_utc", "end_utc", "duration_hours"))

# Lost volume: mean running rate in the lookback window before the event, or the month's mean if there is none
lookback_seconds = DT["lost_volume_lookback_hours"] * 3600
rates = well_hours.filter(F.col("running_rate").isNotNull()).select("well_key", F.col("ts_utc").alias("rate_ts"), "running_rate")
before = (islands.join(rates, "well_key")
    .filter((F.col("rate_ts") < F.col("start_utc"))
            & (F.unix_timestamp("rate_ts") >= F.unix_timestamp("start_utc") - lookback_seconds))
    .groupBy("well_key", "start_utc").agg(F.avg("running_rate").alias("lookback_rate")))
month_rate = rates.groupBy("well_key").agg(F.avg("running_rate").alias("month_rate"))
events = (islands.join(before, ["well_key", "start_utc"], "left").join(month_rate, "well_key", "left")
    .withColumn("rate_used", F.coalesce("lookback_rate", "month_rate"))
    .withColumn("lost_volume", F.col("rate_used") * F.col("duration_hours") / 24)
    .withColumn("event_id", F.concat_ws("|", "well_key", F.date_format("start_utc", "yyyy-MM-dd'T'HH:mm:ss'Z'")))
    .withColumn("_run_id", F.lit(RUN_ID))
    .select("event_id", "well_key", "facility_id", "product", "start_utc", "end_utc", "duration_hours",
            "rate_used", "lost_volume", "_run_id"))
events.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(L["downtime_table"])


In [ ]:
# Monthly sensor volume against Petrinex. An hour with no Good flow reading that is not shut in is filled from the mean
# running rate of the same local day, or the month's if that day has none
filled = (well_hours
    .withColumn("day_rate", F.avg("running_rate").over(Window.partitionBy("well_key", "local_day")))
    .withColumn("month_rate", F.avg("running_rate").over(Window.partitionBy("well_key")))
    .withColumn("hour_volume", F.when(F.col("good_flow"), F.col("value")).when(F.col("shut_in"), 0.0)
                                .otherwise(F.coalesce("day_rate", "month_rate")) / 24)
    .withColumn("is_filled", ~F.col("good_flow") & ~F.col("shut_in")))
sensor_volume = filled.groupBy("well_key", "facility_id", "product").agg(
    F.sum("hour_volume").alias("sensor_volume"),
    F.sum(F.when(F.col("is_filled"), 1).otherwise(0)).alias("hours_filled"))

month_start = CFG["month"]["production_month"] + "-01"
reported = (spark.table(L["production_fact_table"]).filter(F.col("production_month") == month_start)
    .select("well_key", F.col("ProductID").alias("product"), F.col("ReportingFacilityID").alias("facility_id"), "volume", "hours")
    .groupBy("well_key", "product", "facility_id")
    .agg(F.sum("volume").cast("double").alias("reported_volume"), F.max("hours").alias("reported_hours")))
downtime = spark.table(L["downtime_table"]).groupBy("well_key").agg(
    F.sum("duration_hours").alias("sensor_downtime_hours"), F.count(F.lit(1)).alias("downtime_events"))
threshold = CFG["reconciliation"]["threshold_pct"]
reconciliation = (sensor_volume.join(reported, ["well_key", "product", "facility_id"], "left").join(downtime, "well_key", "left")
    .withColumn("difference", F.col("sensor_volume") - F.col("reported_volume"))
    .withColumn("difference_pct", F.col("difference") / F.col("reported_volume") * 100)
    .withColumn("needs_investigation", F.coalesce(F.abs("difference_pct") > threshold, F.lit(True)))
    .withColumn("sensor_downtime_hours", F.coalesce("sensor_downtime_hours", F.lit(0)))
    .withColumn("downtime_events", F.coalesce("downtime_events", F.lit(0)))
    .withColumn("production_month", F.to_date(F.lit(month_start)))
    .withColumn("_run_id", F.lit(RUN_ID))
    .select("production_month", "well_key", "facility_id", "product", "sensor_volume", "reported_volume", "difference",
            "difference_pct", "hours_filled", "sensor_downtime_hours", "downtime_events", "reported_hours",
            "needs_investigation", "_run_id"))
reconciliation.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(L["reconciliation_table"])


In [ ]:
# The 4 reconciliation tests
bronze_rows = (spark.table(L["bronze_backfill_table"]).count() + spark.table(L["bronze_stream_table"]).count())
received_rows = silver.filter("is_received").count()
reject_rows = (spark.table(L["rejects_table"]).filter(F.col("run_id") == RUN_ID).count()
               if spark.catalog.tableExists(L["rejects_table"]) else 0)

gold_hourly = spark.table(L["hourly_table"])
silver_by_tag = silver.filter(F.col("quality") == "Good").groupBy("tag_name").agg(
    F.count(F.lit(1)).alias("s_count"), F.sum("value").alias("s_sum"))
hourly_by_tag = gold_hourly.groupBy("tag_name").agg(F.sum("good_count").alias("g_count"), F.sum("sum_good_value").alias("g_sum"))
hourly_mismatch = (silver_by_tag.join(hourly_by_tag, "tag_name", "full_outer")
    .filter((F.coalesce("s_count", F.lit(0)) != F.coalesce("g_count", F.lit(0)))
            | (F.abs(F.coalesce("s_sum", F.lit(0)) - F.coalesce("g_sum", F.lit(0))) > 0.001))
    .count())

with open(L["truth_file"]) as fh:
    truth_events = json.load(fh)["downtime_events"]
truth = (spark.createDataFrame([(e["well_key"], e["start_utc"], e["hours"]) for e in truth_events],
                               "well_key string, start_utc string, duration_hours long")
    .withColumn("start_utc", F.col("start_utc").cast("timestamp")))
detected = spark.table(L["downtime_table"]).select("well_key", "start_utc", F.col("duration_hours").cast("long"))
event_differences = detected.exceptAll(truth).count() + truth.exceptAll(detected).count()

configured = {a["well_key"] for a in CFG["anomaly_wells"]["wells"]}
gold_rec = spark.table(L["reconciliation_table"])
flagged = {r["well_key"] for r in gold_rec.filter("needs_investigation").select("well_key").collect()}
well_differences = len(configured ^ flagged)

tests = [
    ("bronze_equals_silver_received_plus_rejects", bronze_rows, received_rows + reject_rows),
    ("gold_hourly_good_count_and_sum_match_silver", 0, hourly_mismatch),
    ("downtime_events_match_simulation_truth", 0, event_differences),
    ("flagged_wells_match_configured_anomalies", 0, well_differences),
]
rows = [(RUN_ID, n, int(e), int(a), "PASS" if e == a else "FAIL") for n, e, a in tests]
schema = "run_id string, test_name string, expected long, actual long, status string"
spark.createDataFrame(rows, schema).write.format("delta").mode("append").saveAsTable(L["reconciliation_results_table"])
for r in rows:
    print(r[4], r[1], "expected", r[2], "actual", r[3])
print("truth events:", len(truth_events), "| flagged:", sorted(flagged), "| configured:", sorted(configured))

# Stop on a failed test, after the results are stored
failed = [r for r in rows if r[4] == "FAIL"]
if failed:
    detail = "; ".join(f"{name} (expected {exp}, actual {act})" for _, name, exp, act, _ in failed)
    raise ValueError(f"Reconciliation failed for run {RUN_ID}: {len(failed)} of {len(rows)} tests FAIL: {detail}. "
                     f"See {L['reconciliation_results_table']}. The Gold tables were already overwritten by this run.")
